# VaaniTriage — Phase 1 data generation (free Kaggle GPU)

Runs the three LLM steps that can't run on a laptop, at **zero cost**:

| Step | Model | Where |
|---|---|---|
| 1a. Generate ~6,000 training complaints | Gemini (free tier API) | Google |
| 1b. Blind second labeller, drop disagreements | Qwen2.5-7B-Instruct (open weights) | this GPU |
| 1d. Generate 350 test candidates (different model + prompts) | Qwen2.5-7B-Instruct | this GPU |

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **On**; Add-ons → Secrets → add `GEMINI_API_KEY`.
Every step caches each reply, so if a quota or session limit stops it, just run the cell again (even the next day).

In [ ]:
REPO = "https://github.com/YOUR_GITHUB_USERNAME/vaani-triage.git"   # <- edit
!git clone -q $REPO /kaggle/working/vaani-triage
%cd /kaggle/working/vaani-triage
# Kaggle already has torch/transformers; install only what's missing (no torch reinstall).
!pip install -q -e . --no-deps && pip install -q regex google-genai==2.33.0

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
!python -c "import torch; print(torch.cuda.device_count(), 'GPU(s)')"

## (Optional) resume from a previous session
If you downloaded `phase1_outputs.zip` earlier, upload it as a Kaggle dataset and unzip it here so cached batches are skipped.

In [ ]:
# !unzip -o -q /kaggle/input/YOUR_DATASET/phase1_outputs.zip -d /kaggle/working/vaani-triage

## 1a. Training data with Gemini
First cell prints the plan (calls, tokens, cost = free tier). Second cell runs it.

In [ ]:
!python -m vaani.data.generate_synthetic

In [ ]:
!python -m vaani.data.generate_synthetic --yes

## 1b. Independent labeller (open model on this GPU)

In [ ]:
!python -m vaani.data.validate_labels --yes
!cat data/interim/validation_report.json

## 1d. Test candidates (different model + different prompts)

In [ ]:
!python -m vaani.data.generate_test --yes

## Package outputs
Download `phase1_outputs.zip` from the Output panel and send it back (or commit `data/manual/test_synthetic_review.csv` to your repo). Then review the CSV by hand.

In [ ]:
!zip -q -r /kaggle/working/phase1_outputs.zip data/raw data/interim data/manual -x 'data/*/smoke/*'
!ls -lh /kaggle/working/phase1_outputs.zip